# Audio AI: Speech-to-Text, Retrieval, and Text-to-Speech

This notebook demonstrates a modular audio AI pipeline:

```text
Audio file
-> preprocessing
-> speech-to-text
-> transcript embedding
-> retrieval over past calls
-> LLM answer
-> text-to-speech
-> spoken output
```

The key teaching point: speech-to-text gives us text, embeddings make spoken content searchable, an LLM generates an answer from retrieved context, and text-to-speech turns the answer back into audio.


## What students should understand

By the end of the demo, students should be able to explain:

- why audio needs preprocessing before transcription
- what speech-to-text does and does not do
- why transcript embeddings are useful for business audio
- how retrieval finds similar prior calls
- how an LLM uses transcript plus retrieved context
- why text-to-speech changes the output modality but does not validate correctness


## Setup

Install the Python packages before the session:

```bash
pip install openai-whisper sentence-transformers pandas numpy scikit-learn librosa soundfile pyttsx3 ollama ipython
```

Install FFmpeg before the session:

```bash
# macOS
brew install ffmpeg

# Ubuntu / Debian
sudo apt-get update
sudo apt-get install ffmpeg espeak
```

If you use the local LLM step, pull the model before class:

```bash
ollama pull llama3.2:1b
```

Prepare a short WAV file named `sample_support_call.wav` in the same folder as this notebook.

Recommended spoken content:

```text
Hi, I cannot log into my account. The password reset email never arrives, and I need access today.
```


In [2]:
# Optional install cell.
# Uncomment and run this cell only if the environment is missing packages.

%pip install -q openai-whisper sentence-transformers pandas numpy scikit-learn librosa soundfile pyttsx3 ollama ipython


Note: you may need to restart the kernel to use updated packages.


## Step 1 — Check the environment

This cell checks whether the important Python packages and FFmpeg are available. It does not stop the notebook; it only shows what may need attention.


In [2]:
import importlib.util
import shutil

checks = {
    "librosa": "librosa",
    "soundfile": "soundfile",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "sklearn",
    "sentence_transformers": "sentence_transformers",
    "whisper": "whisper",
    "pyttsx3": "pyttsx3",
    "ollama": "ollama",
}

for label, module_name in checks.items():
    status = "ok" if importlib.util.find_spec(module_name) else "missing"
    print(f"{label:24s} {status}")

print(f"{'ffmpeg':24s} {'ok' if shutil.which('ffmpeg') else 'missing'}")


librosa                  ok
soundfile                ok
pandas                   ok
numpy                    ok
sklearn                  ok
sentence_transformers    ok
whisper                  ok
pyttsx3                  ok
ollama                   ok
ffmpeg                   ok


## Step 2 — Imports and constants

We define the input audio file, the cleaned audio file, and the text-to-speech output file.


In [3]:
from pathlib import Path

INPUT_AUDIO = Path("sample_support_call.wav")
CLEAN_AUDIO = Path("sample_16k_mono.wav")
TTS_OUTPUT = Path("answer.wav")

FALLBACK_TRANSCRIPT = (
    "Hi, I cannot log into my account. "
    "The password reset email never arrives, and I need access today."
)

print("Input audio:", INPUT_AUDIO.resolve())
print("Audio file exists:", INPUT_AUDIO.exists())


Input audio: /home/ml/masterschool/sample_support_call.wav
Audio file exists: False


## Step 3 — Load and preprocess the audio

ASR models usually expect audio in a predictable format. Here we convert the input into **16 kHz mono** audio.

If `sample_support_call.wav` is missing, the notebook continues with a prepared transcript fallback.


In [ ]:
HAS_AUDIO = INPUT_AUDIO.exists()
transcript = None

if HAS_AUDIO:
    import librosa
    import soundfile as sf

    audio, sample_rate = librosa.load(INPUT_AUDIO, sr=16000, mono=True)
    duration = len(audio) / sample_rate

    print("Sample rate:", sample_rate)
    print("Duration:", round(duration, 2), "seconds")
    print("Samples:", audio.shape)

    sf.write(CLEAN_AUDIO, audio, sample_rate)
    print("Saved", CLEAN_AUDIO)
else:
    transcript = FALLBACK_TRANSCRIPT
    print("Missing sample_support_call.wav")
    print("Using fallback transcript instead:")
    print(transcript)


## Step 4 — Listen to the cleaned input audio

This cell plays the cleaned input audio inside the notebook when an audio file is available.


In [ ]:
from IPython.display import Audio, display

if HAS_AUDIO and CLEAN_AUDIO.exists():
    display(Audio(str(CLEAN_AUDIO), autoplay=False))
else:
    print("No input audio available. Continuing with transcript fallback.")


## Step 5 — Transcribe with speech-to-text

Speech-to-text converts spoken language into written text. It is not reasoning; it only produces a transcript.

Everything downstream depends on this transcript being good enough.


In [ ]:
if HAS_AUDIO and CLEAN_AUDIO.exists():
    try:
        import whisper

        asr_model = whisper.load_model("tiny")
        result = asr_model.transcribe(str(CLEAN_AUDIO), fp16=False)
        transcript = result["text"].strip()
    except Exception as exc:
        print("ASR failed. Using fallback transcript instead.")
        print("Error:", repr(exc))
        transcript = FALLBACK_TRANSCRIPT
else:
    transcript = FALLBACK_TRANSCRIPT

print("Transcript:")
print(transcript)


## Step 6 — Build a small retrieval dataset

These are example transcripts from previous support calls. In a production system, each row would usually link to the original audio file, timestamps, speaker metadata, ticket outcome, and permissions.


In [ ]:
import pandas as pd

past_calls = [
    {
        "id": "call_101",
        "topic": "billing",
        "transcript": "The customer was charged twice for the same invoice and wants a refund.",
    },
    {
        "id": "call_102",
        "topic": "password_reset",
        "transcript": "The user cannot log in because the password reset email is not arriving.",
    },
    {
        "id": "call_103",
        "topic": "delivery_delay",
        "transcript": "The customer asks why their package is delayed and wants a new delivery date.",
    },
    {
        "id": "call_104",
        "topic": "audio_issue",
        "transcript": "The microphone is not detected during video calls after a laptop update.",
    },
]

pd.DataFrame(past_calls)


## Step 7 — Embed transcripts and retrieve similar calls

We embed the transcript, not the raw audio.

This is usually the simplest robust approach for speech-first business use cases because it captures **what was said**.

Direct audio embeddings are different. They are useful for speaker similarity, music, environmental sounds, and other acoustic properties.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

past_texts = [item["transcript"] for item in past_calls]

try:
    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    past_embeddings = embedder.encode(
        past_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    query_embedding = embedder.encode(
        [transcript],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    scores = cosine_similarity(query_embedding, past_embeddings)[0]
    retrieval_method = "SentenceTransformer embeddings"

except Exception as exc:
    print("Embedding model failed. Using TF-IDF fallback for demo continuity.")
    print("This fallback is not the same as neural embeddings.")
    print("Error:", repr(exc))

    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(past_texts + [transcript])
    scores = cosine_similarity(matrix[-1], matrix[:-1])[0]
    retrieval_method = "TF-IDF fallback"

results = pd.DataFrame({
    "id": [item["id"] for item in past_calls],
    "topic": [item["topic"] for item in past_calls],
    "transcript": past_texts,
    "score": scores,
}).sort_values("score", ascending=False)

print("Retrieval method:", retrieval_method)
results


## Step 8 — Send transcript and retrieved context to an LLM

The LLM receives the transcript and the retrieved context. It is not listening to the original audio in this pipeline.

This modular design makes the system easier to inspect and debug.


In [ ]:
retrieved_context = results.head(2).to_dict(orient="records")

prompt = f"""
You are a support assistant.

New call transcript:
{transcript}

Similar past calls:
{retrieved_context}

Task:
1. Summarize the user issue in one sentence.
2. Identify the likely intent.
3. Suggest the next best support action.
Keep the answer short and clear.
"""

print(prompt)


In [ ]:
try:
    from ollama import chat

    response = chat(
        model="llama3.2:1b",
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response.message.content.strip()
    llm_method = "Ollama llama3.2:1b"

except Exception as exc:
    print("Ollama unavailable. Using fallback answer.")
    print("Error:", repr(exc))

    answer = (
        "The user cannot log in because the password reset email is not arriving. "
        "Likely intent: account access. "
        "Next action: verify the email address and trigger a manual password reset."
    )
    llm_method = "fallback answer"

print("LLM method:", llm_method)
print()
print(answer)


## Step 9 — Prepare the answer for text-to-speech

Text-to-speech works better when the text is short, clean, and free of Markdown formatting.


In [ ]:
import re


def clean_for_tts(text: str) -> str:
    """Make LLM output easier for a speech engine to read aloud."""
    text = re.sub(r"[*_`#>-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


spoken_answer = clean_for_tts(answer)

# Keep the live voice output short enough for class.
if len(spoken_answer) > 450:
    spoken_answer = spoken_answer[:450] + "."

print(spoken_answer)


## Step 10 — Convert the answer into speech

This is the text-to-speech step. The system is not understanding anything new here. It only changes the output modality from text to audio.

Set `SPEAK_LIVE = True` if you want the system voice to speak immediately through the speakers.


In [ ]:
SPEAK_LIVE = True
SAVE_AUDIO = True

tts_success = False

try:
    import pyttsx3

    engine = pyttsx3.init()
    engine.setProperty("rate", 165)
    engine.setProperty("volume", 0.9)

    if SPEAK_LIVE:
        engine.say(spoken_answer)

    if SAVE_AUDIO:
        engine.save_to_file(spoken_answer, str(TTS_OUTPUT))

    engine.runAndWait()
    tts_success = True

    if SAVE_AUDIO:
        print("Saved", TTS_OUTPUT)
    if SPEAK_LIVE:
        print("Spoken output sent to system voice.")

except Exception as exc:
    print("pyttsx3 failed. The text answer is still available.")
    print("Error:", repr(exc))
    print(spoken_answer)


## Step 11 — Play the generated audio answer

If the WAV file was created successfully, this cell plays it inside the notebook.


In [ ]:
from IPython.display import Audio, display

if TTS_OUTPUT.exists():
    display(Audio(str(TTS_OUTPUT), autoplay=False))
else:
    print("No answer.wav file found. Use the text answer or a platform TTS fallback.")


## Platform TTS fallbacks

Use these only if `pyttsx3` is unstable.

First save the answer text:


In [ ]:
Path("answer.txt").write_text(spoken_answer, encoding="utf-8")
print("Saved answer.txt")


macOS:

```bash
say -o answer.aiff -f answer.txt
```

Ubuntu / Debian:

```bash
espeak -w answer.wav -f answer.txt
```

Windows PowerShell:

```powershell
Add-Type -AssemblyName System.Speech
$speak = New-Object System.Speech.Synthesis.SpeechSynthesizer
$speak.SetOutputToWaveFile("answer.wav")
$speak.Speak((Get-Content answer.txt -Raw))
$speak.Dispose()
```


## Complete pipeline map

```text
sample_support_call.wav
-> sample_16k_mono.wav
-> transcript
-> transcript embedding
-> similar past calls
-> LLM answer
-> answer.wav
```

Ask students:

> Where could this system fail?

Good answers:

- the microphone recording is noisy
- ASR mishears a key word, code, name, or number
- retrieval returns the wrong past call
- the LLM overgeneralizes from weak context
- TTS makes the answer sound more certain than it should


## Student mini challenge

Change the new transcript and rerun retrieval, LLM answer, and TTS.

Example:

```text
My package is delayed and nobody can tell me when it will arrive.
```

Question:

> Did the top retrieved call change from password reset to delivery delay?


In [ ]:
challenge_transcript = "My package is delayed and nobody can tell me when it will arrive."

try:
    challenge_embedding = embedder.encode(
        [challenge_transcript],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    challenge_scores = cosine_similarity(challenge_embedding, past_embeddings)[0]
except Exception:
    # Works with the TF-IDF fallback path if the embedding model was unavailable.
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(past_texts + [challenge_transcript])
    challenge_scores = cosine_similarity(matrix[-1], matrix[:-1])[0]

challenge_results = pd.DataFrame({
    "id": [item["id"] for item in past_calls],
    "topic": [item["topic"] for item in past_calls],
    "transcript": past_texts,
    "score": challenge_scores,
}).sort_values("score", ascending=False)

challenge_results


## Instructor wrap-up

| Step | What it does | What it does not do |
|---|---|---|
| ASR | Converts speech to text | Does not understand intent |
| Transcript embedding | Makes spoken content searchable | Does not reason about the issue |
| Retrieval | Finds similar prior content | Can retrieve irrelevant context |
| LLM | Summarizes and suggests action | Can hallucinate or overstate certainty |
| TTS | Speaks the answer | Does not improve correctness |

Final sentence to students:

> A voice AI system is not one model. It is a pipeline, and each conversion step needs to be tested separately.
